In [ ]:
import pandas as pd
from common import *

from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif

In [ ]:
main_dir = "data/home-credit-credit-risk-model-stability/parquet_files/"
main_dir_train = f"{main_dir}train/"

In [ ]:
train_base = pd.read_parquet(f"{main_dir_train}train_base.parquet")
train_base = train_base[["case_id","target"]]

In [ ]:
# Static
static_names = ["train_static_0_0", "train_static_0_1", "train_static_cb_0"]
train_static = []
for static in static_names:
    train_static.append(pd.read_parquet(f"{main_dir_train}{static}.parquet"))
train_static_a = pd.concat(train_static[:2])
train_static_b = train_static[-1]
del train_static

In [ ]:
# train_static = pd.merge(train_static_a, train_static_b, on=["case_id"],
#                         how="outer")
# train_static = train_static[train_static.columns.sort_values()]

### Static a

In [ ]:
train_static_a = train_static_a[["case_id"] + train_static_a.columns.difference(["case_id"]).sort_values().tolist()]
train_static_a["lastapplicationdate_877D"] = pd.to_datetime(train_static_a["lastapplicationdate_877D"])
train_static_a = train_static_a.replace({"a55475b1":pd.NA})
train_static_a.head()

In [ ]:
features_a = get_column_descriptions(train_static_a)

In [ ]:
plot_null_percent(train_static_a)
train_static_a = delete_null_columns(train_static_a, 0.25)
features_a = get_column_descriptions(train_static_a)

In [ ]:
train_static_a = train_base.merge(train_static_a, on="case_id")

In [ ]:
train_static_a.dtypes.value_counts()

In [ ]:
categorical_v = train_static_a.columns[(train_static_a.dtypes == "object") | (train_static_a.dtypes == "bool")]
import seaborn as sns
categorical_selection = []
for variable in categorical_v:
    counts = train_static_a[variable].value_counts(normalize=True).reset_index()
    plt.figure(figsize=(25,5))
    plt.subplot(1,2,1)
    sns.barplot(data=counts, x=variable, y="proportion")
    counts = train_static_a.groupby(["target"])[variable].value_counts(normalize=True).reset_index()
    plt.subplot(1,2,2)
    sns.barplot(data=counts, x=variable, y="proportion", hue="target")
    data_plot = train_static_a.groupby(["target"])[variable].value_counts(normalize=True).reset_index()

    contingency = pd.crosstab(index=train_static_a["target"],
                columns = [train_static_a[variable]])
    statistic, pvalue, dof, expected_freq = chi2_contingency(contingency.values)
    # print(f"{variable}: {pvalue}")
    plt.suptitle(f"Chi-square {variable}: p-value = {pvalue}")
    plt.show()

    if pvalue < 0.05:
        categorical_selection.append(variable)
categorical_selection

In [ ]:
numerical_cols = [x for x in train_static_a.columns if train_static_a[x].dtype in ["float64", "int64"]]
features_a_numerical = get_column_descriptions(train_static_a[numerical_cols])
# train_static_a[numerical_cols].drop(columns=["case_id","target"]).corr()

# for col in numerical_cols:
#     plt.figure(figsize=(25,5))
#     plt.subplot(1,2,1)
#     sns.boxplot(data=train_static_a, x=col, hue="target")
#     plt.subplot(1,2,2)
#     sns.kdeplot(train_static_a, x=col,
#                 hue="target", common_norm=False)
#     plt.title(f"Variable: {col}")
#     plt.show()

# sns.pairplot(data=train_static_a[numerical_cols].drop(columns=["case_id","target"]))

In [ ]:
from scipy.stats import mannwhitneyu
bests = []
for col in numerical_cols:
    statistic, pvalue = mannwhitneyu(train_static_a[col], train_static_a["target"])
    print(f"Variable: {col}. p-value: {pvalue}")
    if pvalue < 0.05:
        bests.append(col)

Train static b

In [ ]:
train_static_b.head()

In [ ]:
for col in train_static_b.columns:
    if "date" in col:
        train_static_b[col] = pd.to_datetime(train_static_b[col], format="%Y-%m-%d")
train_static_b.dtypes.value_counts()

In [ ]:
get_column_descriptions(train_static_b)

In [ ]:
plot_null_percent(train_static_b)
delete_null_columns(train_static_b, 0.1)

In [ ]:
def mix_cols(row: pd.DataFrame, original_cols: list, new_col_names: list):
    different_vals = []
    for new_col_name, original_cols_set in zip(new_col_names, original_cols):
        row_cols = row[original_cols_set].dropna()
        if not len(row_cols):
            row[new_col_name] = pd.NA
        else:
            unique_values = row_cols.unique()
            row_cols = row_cols.astype(str) if type(unique_values[0]) == str else row_cols
            if len(unique_values) > 1:
                value_counts = row_cols.value_counts(normalize=True)
                majority = value_counts[value_counts>0.5]
                if not majority.empty:
                    row[new_col_name] = majority.iloc[0]
                else:
                    if type(unique_values[0]) == str:
                        row[new_col_name] = unique_values[0]
                    else:
                        try:
                            row_cols = pd.to_datetime(row_cols)
                        except:
                            pass
                        try:
                            row[new_col_name] = row_cols.mean()
                        except:
                            row[new_col_name] = unique_values[0]
                            print(row["case_id"])

                different_vals.append(new_col_name)
            else:
                row[new_col_name] = unique_values[0]
    if not len(different_vals):
        row["different_vals"] = pd.NA
    else:
        row["different_vals"] = ", ".join(different_vals)
    print(row["case_id"])
    return row[["case_id", "different_vals"] + new_col_names]


def mix_cols_parallel(data: pd.DataFrame, original_cols: list):
    list_cols = []
    new_values = []
    for col in original_cols:
        list_cols.append(data[col].tolist())
    for i in range(len(list_cols[0])):
        row_cols = pd.Series([list_cols[x][i] for x in range(len(list_cols))])
        row_cols = row_cols.dropna()
        if not len(row_cols):
            new_values.append(pd.NA)
        else:
            unique_values = row_cols.unique()
            if len(unique_values) > 1:
                value_counts = row_cols.value_counts(normalize=True)
                majority = value_counts[value_counts>0.5]
                if not majority.empty:
                    new_values.append(majority.iloc[0])
                else:
                    if type(unique_values[0]) == str:
                        new_values.append(unique_values[0])
                    else:
                        try:
                            row_cols = pd.to_datetime(row_cols)
                        except:
                            pass
                        try:
                            new_values.append(row_cols.mean())
                        except:
                            new_values.append(unique_values[0])
                            print(i)
            else:
                new_values.append(unique_values[0])
        # print(i)
    return new_values

# case_id = 28629
# case_id = 28657
# case_id = 2701434
# case_id = 2701259
# train_static_b[train_static_b["case_id"]==case_id]

In [ ]:
original_col_list = []
# new_cols = ["assignmentdate", "birth", "education", "maritalst", "pmtaverage", "pmtcount", "responsedate"]
new_cols = ["assignmentdate", "birth", "education", "maritalst", "pmtaverage", "pmtcount", "responsedate"]

for new_col in new_cols:
    original_cols = train_static_b.columns[train_static_b.columns.str.contains(new_col)].tolist()
    original_col_list.append(original_cols)

original_col_list

In [ ]:
from joblib import cpu_count, Parallel, delayed
new_data = Parallel(n_jobs=cpu_count(), backend="multiprocessing")(delayed(mix_cols_parallel)(train_static_b, original_cols) for original_cols in original_col_list)

# import time
# new_data = []
# for new_col, original_cols in zip(new_cols, original_col_list):
#     t1 = time.time()
#     new_data.append(mix_cols_parallel(train_static_b, original_cols))
#     t2 = time.time()
#     print(f"{new_col}: {t2-t1} seconds.")

In [ ]:
train_static_b_f = train_static_b[["case_id"]].copy(deep=True)
for new_col, new_col_data in zip(new_cols, new_data):
    train_static_b_f[new_col] = new_col_data
# del new_data

In [ ]:
import itertools
original_cols = list(itertools.chain.from_iterable(original_col_list))
other_cols = train_static_b.columns.difference(original_cols).tolist()
train_static_b = train_static_b[other_cols]

In [ ]:
# train_static_b.columns = ["a"] + train_static_b.columns.tolist()[1:]

In [ ]:
train_static_b_f = train_static_b_f.merge(train_static_b, on=["case_id"])

In [ ]:
plot_null_percent(train_static_b_f)
train_static_b_f = delete_null_columns(train_static_b_f, 0.1)

In [ ]:
train_static_b_f = train_static_b_f.merge(train_base, on=["case_id"])
for col in ["birth", "responsedate"]:
    train_static_b_f[col] = pd.to_datetime(train_static_b_f[col], format="%Y-%m-%d")

In [ ]:
train_static_b_f.dtypes.value_counts()

In [ ]:
for col in train_static_b_f.columns:
    if train_static_b_f[col].dtype in ["object","categorical"]:
        counts = train_static_b_f[col].value_counts(normalize=True).reset_index()
        plt.figure(figsize=(25,5))
        plt.subplot(1,2,1)
        sns.barplot(data=counts, x=col, y="proportion")
        counts = train_static_b_f.groupby(["target"])[col].value_counts(normalize=True).reset_index()
        plt.subplot(1,2,2)
        sns.barplot(data=counts, x=col, y="proportion", hue="target")

        contingency = pd.crosstab(index=train_static_b_f["target"],
                                  columns = [train_static_b_f[col]])
        statistic, pvalue, dof, expected_freq = chi2_contingency(contingency.values)
        plt.suptitle(f"Chi-square {col}: p-value = {pvalue}")

        plt.show()

In [ ]:
import numpy as np
a = np.floor(((train_static_b_f["responsedate"] - train_static_b_f["birth"]).dt.days)/365)
# ((train_static_b_f["responsedate"] - train_static_b_f["birth"]).dt.days)/365

In [ ]:
numerical_cols = [x for x in train_static_b_f
                    if train_static_b_f[x].dtype in ["int64","float64"] and
                        x not in ["case_id","target"]]
h = train_static_b_f.dropna()
mutual_info_b = mutual_info_classif(h[numerical_cols], y=h["target"])
sorted(((mutual_info_value, col) for col, mutual_info_value in zip(numerical_cols, mutual_info_b)), reverse=True)

In [ ]:
for col in numerical_cols:
    plt.figure(figsize=(25,5))
    sns.boxplot(data=train_static_b_f, x=col, hue="target")
    plt.show()
    # sns.kdeplot(data=train_static_b_f, x=numerical_cols[0], hue="target")

In [ ]:
corr_data = train_static_b_f[numerical_cols].corr().unstack().reset_index()
corr_data.columns = ["v1","v2","corr"]
def reorder_corr_values(row: pd.DataFrame):
    if row["v1"] > row["v2"]:
        row["v1"], row["v2"] = row["v2"], row["v1"]
    return row
corr_data = corr_data.apply(reorder_corr_values, axis=1)
corr_data = corr_data.drop_duplicates()
corr_data = corr_data[corr_data["v1"]!=corr_data["v2"]]
corr_data[corr_data["corr"]>0.8]

# sns.heatmap(data=train_static_b_f[numerical_cols].corr())

In [ ]:
corr_data[corr_data["corr"]>0.8][["v1","v2"]].unstack().value_counts()
# remove days180?
# do pca to get the correlated features and then delete them=

In [ ]:
train_static_b_f